# Whole-Body Pose Estimation (133 Keypoints)
## Eat The Beat BeatBusterz — Dance Masters 2017

**Model:** RTMW (RTMPose Whole-Body) via rtmlib  
**Keypoints:** 133 total:
- 17 body keypoints (COCO)
- 68 face keypoints
- 42 hand keypoints (21 per hand)
- 6 foot keypoints

**Why whole-body for dance?**  
Hand positions and facial expressions are key elements of dance performance.  
Standard 17-keypoint models miss these details entirely.

**Domain shift:** Real competition footage vs studio-controlled AIST++ dataset.

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
!pip install rtmlib --quiet
!pip install opencv-python-headless matplotlib --quiet
 
import torch
!pip uninstall onnxruntime onnxruntime-gpu -y --quiet
!pip install onnxruntime-gpu==1.19.0 --quiet
 
import onnxruntime as ort
providers = ort.get_available_providers()
backend_device = 'cuda' if 'CUDAExecutionProvider' in providers else 'cpu'
print(f'ORT backend: {backend_device}')
print('Done')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
import cv2
import numpy as np
import time
 
VIDEO_PATH  = '/content/drive/MyDrive/Files (1)/BeatBusterz.mp4'
RESULTS_DIR = '/content/drive/MyDrive/BeatBusterz_WB_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('/content/frames_wb', exist_ok=True)
 
cap   = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
 
print(f'Video: {os.path.basename(VIDEO_PATH)}')
print(f'Resolution: {w}x{h}')
print(f'FPS: {fps:.1f}')
print(f'Total frames: {total}')
print(f'Duration: {total/fps:.1f}s = {total/fps/60:.1f}min')


In [ ]:
FRAME_STEP  = max(1, int(fps // 5))
MAX_FRAMES  = 50
 
start_frame = int(total * 0.10)
end_frame   = int(total * 0.90)
 
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
 
frame_paths = []
frame_idx   = start_frame
saved       = 0
 
while cap.isOpened() and saved < MAX_FRAMES and frame_idx < end_frame:
    ret, frame = cap.read()
    if not ret:
        break
    if (frame_idx - start_frame) % FRAME_STEP == 0:
        path = f'/content/frames_wb/frame_{saved:04d}.jpg'
        cv2.imwrite(path, frame)
        frame_paths.append(path)
        saved += 1
    frame_idx += 1
 
cap.release()
print(f'Extracted {saved} frames')
print(f'Time range: {start_frame/fps:.1f}s — {end_frame/fps:.1f}s')


In [ ]:
from rtmlib import Wholebody, draw_skeleton
 
wb_model = Wholebody(
    mode='balanced',
    to_openpose=False,
    backend='onnxruntime',
    device=backend_device
)
 
print(f'Wholebody model loaded on {backend_device.upper()}')
print('Keypoints: 133 (body + face + hands + feet)')


In [ ]:
import matplotlib.pyplot as plt
 
test_path = frame_paths[min(10, len(frame_paths)-1)]
img = cv2.imread(test_path)
 
t0 = time.time()
keypoints, scores = wb_model(img)
t  = time.time() - t0
 
print(f'Inference time: {t*1000:.0f}ms | FPS: {1/t:.1f}')
print(f'Detected people: {len(keypoints)}')
if len(keypoints) > 0:
    print(f'Keypoints per person: {keypoints[0].shape}')
    print(f'  Expected: (133, 2)')
 
img_show = img.copy()
img_show = draw_skeleton(img_show, keypoints, scores, kpt_thr=0.3)
img_rgb  = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)
 
plt.figure(figsize=(14, 8))
plt.imshow(img_rgb)
plt.title(f'Whole-Body 133 Keypoints — {len(keypoints)} dancer(s) detected', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.savefig('wb_test.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
all_stats  = []
all_times  = []
 
print(f'Running Wholebody inference on {len(frame_paths)} frames...')
 
for i, path in enumerate(frame_paths):
    img = cv2.imread(path)
 
    t0 = time.time()
    keypoints, scores = wb_model(img)
    t  = time.time() - t0
    all_times.append(t)
 
    n_det  = len(keypoints) if keypoints is not None else 0
    avg_sc = float(np.mean(scores)) if n_det > 0 else 0.0
 
    all_stats.append({
        'frame':  i,
        'n_det':  n_det,
        'conf':   avg_sc,
        'time':   t
    })
 
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(frame_paths)} | {n_det} detected | {t*1000:.0f}ms')
 
avg_fps  = 1 / np.mean(all_times)
avg_det  = np.mean([s['n_det'] for s in all_stats])
avg_conf = np.mean([s['conf']  for s in all_stats])
 
print(f'\nResults:')
print(f'  FPS:              {avg_fps:.1f}')
print(f'  Avg detections:   {avg_det:.1f} per frame')
print(f'  Avg confidence:   {avg_conf:.3f}')


In [ ]:
from ultralytics import YOLO
 
yolo_model = YOLO('yolo26s-pose.pt')
print('YOLO26s loaded for comparison')


In [ ]:
start    = int(len(frame_paths) * 0.2)
step     = max(1, (len(frame_paths) - start) // 10)
selected = frame_paths[start::step][:10]
 
SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(6,8),(7,9),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(12,14),(13,15),(14,16)
]
COLORS = ['#FF0000','#00FF00','#0000FF','#FF00FF','#FFFF00',
          '#00FFFF','#FF8000','#8000FF','#FF0080','#80FF00']
 
fig, axes = plt.subplots(10, 2, figsize=(16, 50))
fig.suptitle('17 Keypoints (YOLO26s) vs 133 Keypoints (RTMW)\n'
             'BeatBusterz HipHop — Dance Masters 2017',
             fontsize=14, y=1.005)
 
axes[0, 0].set_title('YOLO26s — 17 Keypoints\n(Body only)', fontsize=11)
axes[0, 1].set_title('RTMW — 133 Keypoints\n(Body + Face + Hands + Feet)', fontsize=11)
 
for row, frame_path in enumerate(selected):
    img_bgr = cv2.imread(frame_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
 
    
    axes[row, 0].imshow(img_rgb)
    res = yolo_model(frame_path, verbose=False)
    n_y = 0
    if res[0].keypoints is not None:
        for p_idx in range(len(res[0].keypoints)):
            color = COLORS[p_idx % len(COLORS)]
            kpts  = res[0].keypoints[p_idx].xy[0].cpu().numpy()
            confs = res[0].keypoints[p_idx].conf[0].cpu().numpy()
            for i, j in SKELETON_PAIRS:
                if confs[i] > 0.3 and confs[j] > 0.3:
                    axes[row, 0].plot([kpts[i][0], kpts[j][0]],
                                      [kpts[i][1], kpts[j][1]],
                                      color=color, linewidth=1.5)
            for k in range(17):
                if confs[k] > 0.3:
                    axes[row, 0].scatter(kpts[k][0], kpts[k][1],
                                        c=color, s=15, zorder=5)
            n_y += 1
    axes[row, 0].set_ylabel(f'Frame {start+row*step+1}\n{n_y} detected', fontsize=7)
    axes[row, 0].axis('off')
 
    
    keypoints, scores = wb_model(img_bgr)
    img_show = img_bgr.copy()
    img_show = draw_skeleton(img_show, keypoints, scores, kpt_thr=0.3)
    img_show_rgb = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)
    axes[row, 1].imshow(img_show_rgb)
    n_w = len(keypoints) if keypoints is not None else 0
    axes[row, 1].set_ylabel(f'{n_w} detected\n133 kpts each', fontsize=7)
    axes[row, 1].axis('off')
 
plt.tight_layout()
plt.savefig('wb_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Comparison saved')


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()
 
fig.suptitle('Whole-Body Pose Estimation (133 Keypoints) — BeatBusterz HipHop\n'
             'Body + Face + Hands + Feet', fontsize=14)
 
for i, (ax, frame_path) in enumerate(zip(axes, selected)):
    img_bgr = cv2.imread(frame_path)
    keypoints, scores = wb_model(img_bgr)
 
    img_show = img_bgr.copy()
    img_show = draw_skeleton(img_show, keypoints, scores, kpt_thr=0.3)
    img_rgb  = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)
 
    ax.imshow(img_rgb)
    n = len(keypoints) if keypoints is not None else 0
    ax.set_title(f'Frame {i+1} — {n} dancers', fontsize=9)
    ax.axis('off')
 
plt.tight_layout()
plt.savefig('wb_wholebody_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grid saved')


In [ ]:
import pandas as pd
import shutil
 
summary = pd.DataFrame([{
    'Model':           'RTMW (Wholebody)',
    'Keypoints':       133,
    'Covers':          'Body + Face + Hands + Feet',
    'Avg Detections':  round(avg_det, 2),
    'Avg Confidence':  round(avg_conf, 3),
    'FPS':             round(avg_fps, 1),
    'Backend':         backend_device.upper()
}])
 
print('=== WHOLE-BODY RESULTS ===')
print(summary.to_string(index=False))
summary.to_csv('wb_summary.csv', index=False)
 
files = [
    'wb_test.png',
    'wb_comparison.png',
    'wb_wholebody_grid.png',
    'wb_summary.csv'
]
 
for f in files:
    if os.path.exists(f):
        shutil.copy(f, f'{RESULTS_DIR}/{f}')
        print(f'Saved: {f}')
 
print(f'\nAll results → BeatBusterz_WB_Results/')
